# Topic 4 → Missing Data Visualization

## Subtopic 4.1 → Visualizing Missing Patterns

---

# Short Definition

> **Visualizing Missing Patterns means identifying where missing values occur and whether they follow any pattern.**
---

# Why do we use it?

It helps answer questions like:

* Are missing values randomly distributed?
* Do certain columns tend to be missing together?
* Do some rows have many missing values?
* Is the missingness informative?

---

# Intuition

Suppose:

| Row | A | B | C |
| --- | - | - | - |
| 1   | ✓ | ✗ | ✗ |
| 2   | ✓ | ✓ | ✓ |
| 3   | ✗ | ✗ | ✓ |
| 4   | ✓ | ✗ | ✗ |

You may notice:

> Whenever `B` is missing, `C` is also missing.

This pattern can be useful.

---

# 1. Missing Value Heatmap

## Short Definition

> Shows the location of missing values using colors.

---

## Entire Syntax

```python
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

sns.heatmap(
    df.isnull(),
    cbar=False
)

plt.show()
```

---

# Important Parameters

| Parameter | Meaning                | Values                 | Default  |
| --------- | ---------------------- | ---------------------- | -------- |
| `data`    | Boolean missing matrix | `df.isnull()`          | Required |
| `cbar`    | Show color bar         | `True`, `False`        | `True`   |
| `cmap`    | Color palette          | `"viridis"`, `"Blues"` | Default  |

---

# Interpretation

Suppose:

```text
██ = Missing
░░ = Present
```

```text
Row1   ░░ ██ ██
Row2   ░░ ░░ ░░
Row3   ██ ██ ░░
```

You can visually spot:

* Missing clusters
* Columns with many missing values
* Rows with many missing values

---

# Common Mistake

❌ Using this on datasets with hundreds of columns.

It becomes unreadable.

---

# 2. Missing Bar Plot

## Short Definition

> A bar plot showing the number of missing values in each column.

---

## Entire Syntax

```python
missing = df.isnull().sum()

missing[
    missing > 0
].sort_values(
    ascending=False
).plot(
    kind="bar",
    figsize=(12,6)
)

plt.ylabel("Missing Count")
plt.show()
```

---

# Interpretation

Tall bars:

> More missing values.

Short bars:

> Fewer missing values.

---

# 3. Missingness Together

Sometimes you discover:

```text
MasVnrType missing
↓
MasVnrArea also missing
```

or

```text
GarageType missing
↓
GarageYrBlt also missing
```

This means:

> Missing values are NOT random.

They occur together because of a real-world reason.

---

# House Prices Example

You observed:

```text
MasVnrType
MasVnrArea
```

If a house has:

> No masonry veneer,

then both become missing.

Similarly:

```text
GarageType
GarageYrBlt
GarageFinish
```

Missing together often means:

> The house has no garage.

---

# How to Interpret Patterns

## Random Missingness

```text
✓ ✗ ✓
✗ ✓ ✗
✓ ✓ ✗
```

Means:

> Missing values are scattered randomly.

---

## Structured Missingness

```text
✓ ✗ ✗
✓ ✗ ✗
✓ ✗ ✗
```

Means:

> Missing values follow a pattern.

Investigate why.

---

## Entire Column Missing

```text
✗
✗
✗
✗
```

Means:

> The feature may not be useful.

# Topic 4 → Missing Data Visualization

## Subtopic 4.2 → Informative Missingness (Very Important)

---

# Short Definition

> **Informative Missingness means that the fact that a value is missing itself contains useful information.**

Usually, we think:

> Missing = Problem.

But sometimes:

> Missing = Information.

---

# Why do we use it?

Because blindly imputing missing values can destroy valuable information.

It helps answer questions like:

* Does missing mean "not applicable"?
* Is missing related to the target?
* Should missing be treated as its own category?
* Should we create a missing indicator feature?

---

# Intuition

Imagine a form with this question:

```text
Car Registration Number
```

Suppose some people leave it blank.

Two possibilities:

### Case 1

They forgot to fill it.

```text
Missing = Accident
```

---

### Case 2

They don't own a car.

```text
Missing = Useful Information
```

The missingness itself tells you something.

---

# House Prices Examples

These are the most famous examples.

---

## Example 1 → PoolQC

Feature:

```text
PoolQC
```

Meaning:

> Pool Quality

Values:

```text
Ex
Gd
TA
Fa
NaN
```

Question:

Why is it missing?

Usually:

> The house has NO pool.

So:

```text
NaN ≠ Unknown

NaN = No Pool
```

This is informative.

---

## Example 2 → FireplaceQu

Feature:

```text
FireplaceQu
```

Meaning:

> Fireplace Quality

Missing often means:

> No fireplace exists.

---

## Example 3 → Garage Features

Features:

```text
GarageType
GarageFinish
GarageQual
GarageCond
GarageYrBlt
```

Missing together often means:

> The house has NO garage.

---

# Why is this Important?

Suppose you do:

```python
df["PoolQC"].fillna(
    df["PoolQC"].mode()[0]
)
```

You replace:

```text
NaN
↓
Gd
```

Now the model thinks:

> Houses without pools have good-quality pools.

Which is completely wrong.

---

# Better Approach

Instead of:

```text
NaN
↓
Most Frequent Value
```

Do:

```text
NaN
↓
"None"
```

Example:

```python
df["PoolQC"] = df["PoolQC"].fillna(
    "None"
)
```

Now:

```text
Ex
Gd
TA
Fa
None
```

The model understands:

> This house simply doesn't have a pool.

---

# Missing Indicator Feature

Sometimes we create a new feature.

Suppose:

```text
LotFrontage
```

Missing may indicate something useful.

We create:

```python
df["LotFrontage_missing"] = (
    df["LotFrontage"]
    .isnull()
    .astype(int)
)
```

---

# Output

Original:

| LotFrontage |
| ----------: |
|          80 |
|         NaN |
|          70 |

New Feature:

| LotFrontage_missing |
| ------------------: |
|                   0 |
|                   1 |
|                   0 |

Interpretation:

```text
0 → Present
1 → Missing
```

---

# Entire Syntax

## Treat Missing as Category

```python
df["PoolQC"] = df["PoolQC"].fillna(
    "None"
)
```

---

## Create Missing Indicator

```python
df["PoolQC_missing"] = (
    df["PoolQC"]
    .isnull()
    .astype(int)
)
```

---

# What Does Each Part Mean?

## `.isnull()`

Returns:

```python
True
False
```

Example:

```text
NaN → True
Ex  → False
```

---

## `.astype(int)`

Converts:

```text
True  → 1
False → 0
```

Output:

```text
Missing     → 1
Not Missing → 0
```

---

# How to Decide if Missingness is Informative?

Ask:

## Question 1

> Does "missing" mean "does not exist"?

Examples:

```text
PoolQC
FireplaceQu
GarageType
Fence
Alley
```

Usually YES.

---

## Question 2

> Is missing related to the target?

Example:

Suppose houses with missing `PoolQC` sell for much lower prices.

Then:

> Missingness itself is predictive.

---

## Question 3

> Could the missing value simply be an error?

Example:

```text
Age = NaN
Salary = NaN
```

Often:

> Missing is not informative.

Use imputation.

---

# Common Mistakes

### Mistake 1

❌ Treating all NaNs the same.

Remember:

```text
Some NaNs = Errors

Some NaNs = Information
```

---

### Mistake 2

❌ Filling informative NaNs with the mode.

Example:

```text
No Pool
↓
Good Pool
```

Wrong.

---

### Mistake 3

❌ Forgetting to create missing indicators.

Sometimes:

> The indicator itself becomes an important feature.